# Transfer Learning Avanzado con YOLOv8: Detección de EPP (Fase 4)

- **Proyecto:** Argus Vision - Sahuaro Data Analytics
- **Autor:** Francisco Ortega Y Jose Cazares
- **Entorno de Ejecución:** Centro de Supercómputo "La Yuca" (GPU AMD Instinct MI210)
- **Dataset:** SH17 (Human Safety & PPE Detection - Pexels)

## Objetivo de la Libreta
Esta libreta documenta el proceso final de entrenamiento (Fine-Tuning) para nuestro sistema de detección de Equipo de Protección Personal (EPP). 

A diferencia de las iteraciones anteriores, aquí implementamos tres mejoras críticas:
1. **Hardware Hibridado:** Utilizaremos la supercomputadora "La Yuca" para reducir drásticamente los tiempos de entrenamiento.
2. **Dataset Industrial Puro:** Migramos al dataset SH17 (vía Kaggle), compuesto por +8,000 imágenes de alta resolución enfocadas estrictamente en entornos de manufactura y construcción.
3. **Data Augmentation Dinámico:** Implementaremos parámetros avanzados en YOLOv8 para forzar a la red neuronal a aprender formas geométricas en lugar de depender exclusivamente de colores o iluminación.

---

## 1. Setup y Detección de Hardware

En el siguiente bloque, importamos las librerías base y definimos una función agnóstica para la detección de hardware. Esto garantiza que el código sea portable y siempre aproveche la aceleración por GPU (ya sea CUDA en servidores Linux, MPS en Apple Silicon, o ROCm para las tarjetas AMD de La Yuca).

In [1]:
import sys
import torch
import subprocess

print("🔍 --- DIAGNÓSTICO PROFUNDO DE LA YUCA --- 🔍")

# 1. ¿A dónde está conectado Jupyter realmente?
print(f"\n1. Ruta del Python actual:\n   {sys.executable}")
if ".venv" in sys.executable:
    print("   Bien: Está usando tu entorno virtual.")
else:
    print("   MAL: Jupyter ignoró tu entorno y está usando el Python genérico.")

# 2. ¿Qué versión de PyTorch cargó?
print(f"\n2. Versión de PyTorch cargada:\n   {torch.__version__}")
if "rocm" in torch.__version__:
    print("   Bien: Es la versión especial para tarjetas AMD.")
else:
    print("   MAL: Es la versión normal/CPU.")

# 3. ¿El nodo "VIP" realmente tiene la GPU conectada físicamente?
print("\n3. Buscando hardware físico (Tarjetas MI210):")
try:
    resultado = subprocess.run(['rocm-smi'], capture_output=True, text=True)
    if resultado.stdout:
        print("   HARDWARE ENCONTRADO! El sistema operativo sí ve las tarjetas.")
    else:
        print("   El sistema no arrojó datos de hardware.")
except Exception as e:
    print("   No se encontró el comando de AMD (rocm-smi). ¿Seguro que estamos en el nodo GPU?")

🔍 --- DIAGNÓSTICO PROFUNDO DE LA YUCA --- 🔍

1. Ruta del Python actual:
   /lustre/cursos/curso04/estudiante_64/epp-detection/.venv/bin/python3
   Bien: Está usando tu entorno virtual.

2. Versión de PyTorch cargada:
   2.11.0+rocm7.2
   Bien: Es la versión especial para tarjetas AMD.

3. Buscando hardware físico (Tarjetas MI210):
   HARDWARE ENCONTRADO! El sistema operativo sí ve las tarjetas.


In [2]:
import torch
from ultralytics import YOLO
import time
import os
import glob
import yaml

def get_optimal_device():
    # En La Yuca con AMD, is_available() es True si ROCm está bien puesto
    if torch.cuda.is_available():
        # Verificamos si es realmente AMD mediante la versión de torch
        version = torch.version.hip if hasattr(torch.version, 'hip') else None
        if version:
            print(f"Hardware detectado: AMD GPU (ROCm {version}).")
        else:
            print("Hardware detectado: CUDA (NVIDIA).")
        return 'cuda'
    elif torch.backends.mps.is_available():
        print("Hardware detectado: MPS (Apple Silicon).")
        return 'mps'
    else:
        print("Advertencia: No se detectó aceleración por hardware. Usando CPU.")
        return 'cpu'

device = get_optimal_device()

Hardware detectado: AMD GPU (ROCm 7.2.26015).


## 1. Descarga del Dataset SH17 (Kaggle)

En esta celda nos conectamos a la API de Kaggle mediante un token de autenticación para descargar el dataset **SH17**. Este es un conjunto de datos masivo (~14 GB) que contiene más de 8,000 imágenes de alta resolución en entornos industriales reales, ya etiquetadas en formato YOLO.

Para optimizar el tiempo y los recursos del servidor, el código incluye una validación: primero verifica si la carpeta destino ya existe y tiene archivos. Si los datos ya fueron descargados en una sesión anterior, omitirá el proceso de descarga automáticamente.

In [3]:
# Ruta donde queremos guardar el dataset
dataset_path = "../data/raw/kaggle/sh17_dataset"

# 1. Configuracion de seguridad (Usando el nuevo Token Directo)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_6a503b1d7288a9b26db05f9286873988"
print("Token de Kaggle configurado directamente en memoria.")

# 2. Validacion de existencia y Descarga
if os.path.exists(dataset_path) and len(os.listdir(dataset_path)) > 0:
    print(f"El dataset ya fue descargado previamente en: {dataset_path}")
    print("Omitiendo la descarga para ahorrar tiempo y recursos.")
else:
    print("La carpeta esta vacia o no existe.")
    print("Iniciando descarga del dataset SH17 (14 GB, paciencia)...")
    
    !uv run kaggle datasets download -d mugheesahmad/sh17-dataset-for-ppe-detection --unzip -p ../data/raw/kaggle/sh17_dataset
    
    print("Descarga y descompresion exitosa.")
    print(f"Tus datos estan listos en la ruta: {dataset_path}")

Token de Kaggle configurado directamente en memoria.
El dataset ya fue descargado previamente en: ../data/raw/kaggle/sh17_dataset
Omitiendo la descarga para ahorrar tiempo y recursos.


## 2. Inicialización del Modelo Base (YOLOv8 Nano)

En esta etapa preparamos el entorno para iniciar el proceso de **Transfer Learning**. 
En lugar de entrenar una red neuronal desde cero, instanciamos el modelo preentrenado `yolov8n.pt` (la versión más ligera y rápida de YOLO). Este modelo ya posee un conocimiento profundo sobre extracción de características visuales (bordes, formas y texturas) adquirido en datasets masivos, lo cual acelerará drásticamente nuestro entrenamiento con el dataset industrial.

Además, vinculamos la configuración estructural de las clases mediante el archivo `data.yaml` generado en el paso anterior.

In [4]:
# Cargamos el modelo "yolov8n.pt"
print("Cargando modelo base YOLOv8n para Transfer Learning...")
model = YOLO('yolov8n.pt') 

# Configurando la ruta del archivo yaml_path
yaml_path = f"{dataset_path}/data.yaml"
print(f"Archivo YAML configurado en: {yaml_path}")

Cargando modelo base YOLOv8n para Transfer Learning...
Archivo YAML configurado en: ../data/raw/kaggle/sh17_dataset/data.yaml


## 3. Generación del Archivo de Configuración (data.yaml)

Para que la arquitectura de YOLOv8 pueda interpretar nuestro dataset, es obligatorio estructurar un archivo de configuración `.yaml`. Este documento actúa como el "mapa" que le indica a la red neuronal dónde encontrar las imágenes de entrenamiento/validación y cómo traducir los números de las etiquetas a texto.

**Decisión Técnica:** En esta celda generamos el archivo dinámicamente mapeando las 17 categorías originales del dataset SH17. Aunque nuestro objetivo comercial en Argus Vision es auditar únicamente un subconjunto de estas (Persona, Casco, Chaleco, Lentes y Guantes), es fundamental entrenar el modelo con el mapeo completo para conservar la integridad geométrica de las etiquetas y evitar errores de índice de clases (*Out of Bounds*). El filtrado final para las alertas se gestionará en la capa de software (Streamlit).

In [5]:
# Ruta base del dataset
dataset_path = "../data/raw/kaggle/sh17_dataset"

# Definir la estructura del archivo YAML para YOLO
data_config = {
    'path': dataset_path,  # Ruta raíz del dataset
    'train': 'images',      # Carpeta de imágenes de entrenamiento
    'val': 'images',        # Carpeta de imágenes de validación
    
    # Nombres de las 17 clases del dataset SH17
    'names': {
        0: 'Person',
        1: 'Head',
        2: 'Face',
        3: 'Glasses',
        4: 'Face-mask-medical',
        5: 'Face-guard',
        6: 'Ear',
        7: 'Earmuffs',
        8: 'Hands',
        9: 'Gloves',
        10: 'Foot',
        11: 'Shoes',
        12: 'Safety-vest',
        13: 'Tools',
        14: 'Helmet',
        15: 'Medical-suit',
        16: 'Safety-suit'
    }
}

# Crear el archivo data.yaml
yaml_path = f"{dataset_path}/data.yaml"
with open(yaml_path, 'w') as file:
    yaml.dump(data_config, file, default_flow_style=False, sort_keys=False)

print(f"✓ Archivo data.yaml creado exitosamente en: {yaml_path}")
print("\nContenido del archivo:")
print("-" * 50)
with open(yaml_path, 'r') as file:
    print(file.read())

✓ Archivo data.yaml creado exitosamente en: ../data/raw/kaggle/sh17_dataset/data.yaml

Contenido del archivo:
--------------------------------------------------
path: ../data/raw/kaggle/sh17_dataset
train: images
val: images
names:
  0: Person
  1: Head
  2: Face
  3: Glasses
  4: Face-mask-medical
  5: Face-guard
  6: Ear
  7: Earmuffs
  8: Hands
  9: Gloves
  10: Foot
  11: Shoes
  12: Safety-vest
  13: Tools
  14: Helmet
  15: Medical-suit
  16: Safety-suit



## 4. Fase 1: Fundación Geométrica (640px)

Esta primera fase de entrenamiento establece la "fundación" del modelo utilizando **Progressive Resizing** (Redimensionamiento Progresivo). Iniciamos el entrenamiento a una resolución estándar de 640x640 píxeles. 

**Objetivos de la Fase 1:**
1. **Aprendizaje de Formas Básicas:** A esta resolución, el modelo no se distrae con detalles hiperfinos y se concentra en aprender las geometrías generales y siluetas del Equipo de Protección Personal (cascos, chalecos, formas humanas).
2. **Robustez mediante Data Augmentation:** Implementamos técnicas moderadas de alteración de imágenes (rotación, cambio de escala, alteraciones de brillo y saturación `hsv`, y técnicas avanzadas como `mosaic` y `erasing`). Esto fuerza a la red a no depender exclusivamente de colores característicos (como el neón de un chaleco), sino a comprender el contexto y la forma del objeto.

**Mecanismo de Resiliencia (HPC):**
Debido a las restricciones de tiempo de sesión en servidores de supercómputo (HPC) como "La Yuca", esta celda cuenta con un sistema de reanudación inteligente. Verifica la existencia del archivo `results.png` para confirmar si la fase concluyó exitosamente. Si fue interrumpida, reanudará automáticamente desde el último punto de guardado (`last.pt`) sin perder el progreso anterior.

In [6]:
# 1. Detectar dónde estamos trabajando (/notebooks)
ruta_actual = os.getcwd()
print(f"Directorio actual: {ruta_actual}")

# 2. Definir las rutas base
ruta_base = os.path.abspath(os.path.join(ruta_actual, '..')) 
ruta_modelos = os.path.join(ruta_base, 'models')

# 3. LA RUTA EXACTA DE TU YAML (Respetando tu celda de configuración)
yaml_path = os.path.join(ruta_base, 'data', 'raw', 'kaggle', 'sh17_dataset', 'data.yaml')

print(f"Ruta base del proyecto: {ruta_base}")
print(f"Ruta de modelos: {ruta_modelos}")
print(f"Ruta del archivo YAML: {yaml_path}")

# Crear la carpeta models por si acaso
os.makedirs(ruta_modelos, exist_ok=True)

Directorio actual: /lustre/cursos/curso04/estudiante_64/epp-detection/notebooks
Ruta base del proyecto: /lustre/cursos/curso04/estudiante_64/epp-detection
Ruta de modelos: /lustre/cursos/curso04/estudiante_64/epp-detection/models
Ruta del archivo YAML: /lustre/cursos/curso04/estudiante_64/epp-detection/data/raw/kaggle/sh17_dataset/data.yaml


In [7]:
print("\n" + "=" * 80)
print("FASE 1/4: FUNDACION - 640px")
print("=" * 80)

fase1_name = 'fase1_fundacion_640px'
fase1_last = f'{ruta_modelos}/{fase1_name}/weights/last.pt'
fase1_best = f'{ruta_modelos}/{fase1_name}/weights/best.pt'
fase1_done = f'{ruta_modelos}/{fase1_name}/results.png' # EL TRUCO ESTÁ AQUÍ

# 1. Si ya se generó results.png, el entrenamiento terminó al 100%
if os.path.exists(fase1_done):
    print("FASE 1 YA COMPLETADA AL 100%. Saltando a la siguiente fase...")

# 2. Si no ha terminado, pero existe last.pt, reanuda
elif os.path.exists(fase1_last):
    print("Reanudando Fase 1 desde el ultimo checkpoint (last.pt)...")
    inicio_fase1 = time.time()
    model_640 = YOLO(fase1_last)
    results_640 = model_640.train(resume=True) 
    tiempo_fase1 = (time.time() - inicio_fase1) / 60
    print(f"\n✓ FASE 1 COMPLETADA en {tiempo_fase1:.2f} minutos")

# 3. Si no hay nada, empieza de cero
else:
    print("Iniciando entrenamiento Fase 1 desde cero...")
    inicio_fase1 = time.time()
    model_640 = YOLO('yolov8n.pt')
    results_640 = model_640.train(
        data=yaml_path,
        epochs=35, 
        imgsz=640,
        device=device,
        project=ruta_modelos,
        name=fase1_name,
        batch=24,
        degrees=10.0, translate=0.15, scale=0.8, perspective=0.0003, fliplr=0.5, flipud=0.0,
        hsv_h=0.01, hsv_s=0.5, hsv_v=0.3,
        mosaic=1.0, mixup=0.1, copy_paste=0.05, erasing=0.3, close_mosaic=10,
        optimizer='AdamW', lr0=0.001, lrf=0.01, momentum=0.937, weight_decay=0.0005,
        warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1, cos_lr=True,
        workers=16, amp=True, cache=False,
        val=True, plots=True, save=True, save_period=5, patience=100, verbose=True
    )
    tiempo_fase1 = (time.time() - inicio_fase1) / 60
    print(f"\n✓ FASE 1 COMPLETADA en {tiempo_fase1:.2f} minutos")


FASE 1/4: FUNDACION - 640px
FASE 1 YA COMPLETADA AL 100%. Saltando a la siguiente fase...


## 5. Fase 2: Refinamiento para Objetos Pequeños (1280px)

Una vez que el modelo ha dominado las formas básicas en la Fase 1, incrementamos la resolución de entrada a 1280x1280 píxeles. Esta fase es crítica para la detección de equipo de protección de menor tamaño.

**Objetivos de la Fase 2:**
1. **Transferencia de Conocimiento:** En lugar de empezar desde cero, cargamos los pesos obtenidos en la Fase 1 (`best.pt`). El modelo ya "sabe" qué es un humano o un chaleco, y ahora utilizará la mayor cantidad de píxeles disponibles para aprender a distinguir detalles finos, como lentes de seguridad (`Glasses`) y guantes (`Gloves`).
2. **Augmentation Agresivo:** Incrementamos significativamente la agresividad de las alteraciones en las imágenes (mayor distorsión de perspectiva, cambios de color más bruscos y borrado de secciones más grandes con `erasing`). Esto simula condiciones extremas en la obra (como sombras fuertes, oclusiones parciales o herramientas tapando parte del equipo) forzando al modelo a ser más preciso.

*Nota: Al igual que las demás fases, esta celda cuenta con el sistema de reanudación automática para tolerar desconexiones del servidor HPC.*

In [8]:
print("\n" + "=" * 80)
print("FASE 2/4: REFINAMIENTO - 1280px")
print("=" * 80)

fase1_best = f'{ruta_modelos}/fase1_fundacion_640px/weights/best.pt'
fase2_name = 'fase2_refinamiento_1280px'
fase2_last = f'{ruta_modelos}/{fase2_name}/weights/last.pt'
fase2_best = f'{ruta_modelos}/{fase2_name}/weights/best.pt'
fase2_done = f'{ruta_modelos}/{fase2_name}/results.png'

if not os.path.exists(fase1_best):
    print("ERROR: No se encontro el modelo 'best.pt' de Fase 1.")

# 1. Checar si ya terminó 100%
elif os.path.exists(fase2_done):
    print("FASE 2 YA COMPLETADA AL 100%. Saltando a la siguiente fase...")

# 2. Reanudar si se cortó a medias
elif os.path.exists(fase2_last):
    print("Reanudando Fase 2 desde el ultimo checkpoint (last.pt)...")
    inicio_fase2 = time.time()
    model_1280 = YOLO(fase2_last)
    results_1280 = model_1280.train(resume=True)
    tiempo_fase2 = (time.time() - inicio_fase2) / 60
    print(f"\n✓ FASE 2 COMPLETADA en {tiempo_fase2:.2f} minutos")

# 3. Empezar Fase 2 nueva
else:
    print("Iniciando entrenamiento Fase 2 desde cero (Usando pesos Fase 1)...")
    inicio_fase2 = time.time()
    model_1280 = YOLO(fase1_best)
    results_1280 = model_1280.train(
        data=yaml_path,
        epochs=40,
        imgsz=1280,
        device=device,
        project=ruta_modelos,
        name=fase2_name,
        batch=10,
        degrees=15.0, translate=0.2, scale=0.7, perspective=0.0005, fliplr=0.5, flipud=0.0,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        mosaic=1.0, mixup=0.15, copy_paste=0.1, erasing=0.4, close_mosaic=15,
        optimizer='AdamW', lr0=0.0005, lrf=0.001, momentum=0.937, weight_decay=0.0005,
        warmup_epochs=5.0, cos_lr=True,
        workers=16, amp=True, cache=False,
        val=True, plots=True, save=True, save_period=5, patience=100, verbose=True
    )
    tiempo_fase2 = (time.time() - inicio_fase2) / 60
    print(f"\n✓ FASE 2 COMPLETADA en {tiempo_fase2:.2f} minutos")


FASE 2/4: REFINAMIENTO - 1280px
FASE 2 YA COMPLETADA AL 100%. Saltando a la siguiente fase...


## 6. Fase 3: Alta Resolución y Robustez Extrema (1920px)

En esta tercera etapa, llevamos al modelo a su máxima capacidad analítica procesando las imágenes a una resolución nativa de 1920x1920 píxeles. Esta fase es la más pesada computacionalmente y tiene como objetivo alcanzar la precisión absoluta en detalles microscópicos.

**Estrategias Clave de la Fase 3:**
1. **Gestión de Memoria (VRAM):** Al cuadruplicar la cantidad de píxeles respecto a la Fase 1, la memoria de la GPU se llena rápidamente. Por ello, es imperativo reducir el tamaño del lote (`batch=6`) para evitar errores de desbordamiento de memoria (*Out of Memory - OOM*) sin sacrificar el aprendizaje.
2. **Caos Controlado (Augmentation Extremo):** En esta etapa activamos las técnicas más agresivas disponibles en YOLOv8. Introducimos `mixup` (fusionar dos imágenes semitransparentes), `copy_paste` (pegar equipo de protección en lugares aleatorios) y llevamos el `erasing` al 50%. Esto fuerza a la red neuronal a detectar un casco o unos lentes incluso si están sucios, tapados por maquinaria, o bajo iluminación extrema, garantizando que el modelo sobreviva a la crudeza visual de una obra de construcción real.

In [9]:
print("\n" + "=" * 80)
print("FASE 3/4: ALTA RESOLUCION - 1920px")
print("Objetivo: Maximizar precision en resolucion nativa con Augmentation Extremo")
print("=" * 80)

fase2_best = f'{ruta_modelos}/fase2_refinamiento_1280px/weights/best.pt'
fase3_name = 'fase3_alta_resolucion_1920px'
fase3_last = f'{ruta_modelos}/{fase3_name}/weights/last.pt'
fase3_best = f'{ruta_modelos}/{fase3_name}/weights/best.pt'
fase3_done = f'{ruta_modelos}/{fase3_name}/results.png'

if not os.path.exists(fase2_best):
    print("[ERROR] No se encontro el modelo 'best.pt' de Fase 2. Ejecute primero la Fase 2.")

# 1. Checar si ya termino al 100%
elif os.path.exists(fase3_done):
    print("[✓] FASE 3 YA COMPLETADA AL 100%. Saltando a la siguiente fase...")

# 2. Reanudar si se corto a medias
elif os.path.exists(fase3_last):
    print("[!] Reanudando Fase 3 desde el ultimo checkpoint (last.pt)...")
    inicio_fase3 = time.time()
    model_1920 = YOLO(fase3_last)
    results_1920 = model_1920.train(resume=True)
    tiempo_fase3 = (time.time() - inicio_fase3) / 60
    print(f"\n[✓] FASE 3 COMPLETADA en {tiempo_fase3:.2f} minutos")

# 3. Empezar Fase 3 nueva
else:
    print("Iniciando entrenamiento Fase 3 desde cero (Usando pesos Fase 2)...")
    inicio_fase3 = time.time()
    model_1920 = YOLO(fase2_best)
    results_1920 = model_1920.train(
        data=yaml_path,
        epochs=50,
        imgsz=1920,
        device=device,
        project=ruta_modelos,
        name=fase3_name,
        batch=6, # Batch muy pequeño porque 1920px consume mucha RAM
        # Augmentation Extremo
        degrees=20.0, translate=0.25, scale=0.6, perspective=0.001, fliplr=0.5, flipud=0.0,
        hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
        mosaic=1.0, mixup=0.2, copy_paste=0.15, erasing=0.5, close_mosaic=20,
        optimizer='AdamW', lr0=0.0002, lrf=0.0001, momentum=0.937, weight_decay=0.0005,
        warmup_epochs=8.0, cos_lr=True,
        workers=16, amp=True, cache=False,
        val=True, plots=True, save=True, save_period=5, patience=150, verbose=True
    )
    tiempo_fase3 = (time.time() - inicio_fase3) / 60
    print(f"\n[✓] FASE 3 COMPLETADA en {tiempo_fase3:.2f} minutos")


FASE 3/4: ALTA RESOLUCION - 1920px
Objetivo: Maximizar precision en resolucion nativa con Augmentation Extremo
[✓] FASE 3 YA COMPLETADA AL 100%. Saltando a la siguiente fase...


## 7. Fase 4: Fine-Tuning Final y Estabilización (1920px)

Esta es la etapa culminante de nuestro pipeline de *Progressive Resizing*. Después de haber sometido a la red neuronal a escenarios extremos en la Fase 3, el objetivo ahora es "enfriar" el modelo (*Cooldown Phase*) para que se estabilice y alcance su máxima precisión antes de salir a producción.

**Estrategias Clave de la Fase 4:**
1. **Reducción del Learning Rate:** Reducimos drásticamente la tasa de aprendizaje (`lr0=0.0001`). Al dar pasos matemáticos mucho más pequeños, evitamos que el modelo "brinque" fuera de su punto óptimo y le permitimos hacer ajustes microscópicos en los pesos neuronales.
2. **Augmentation Mínimo (Realismo puro):** Apagamos casi por completo el caos visual (`mixup=0`, `copy_paste=0`, bajamos el `mosaic` al 30% y casi eliminamos el `erasing`). En las últimas épocas, el modelo necesita ver las imágenes de la forma más natural y limpia posible, exactamente como las verá a través de las cámaras en el dashboard de Argus Vision.
3. **Preparación para Producción:** El archivo resultante de esta fase (`best.pt`) será nuestro modelo maestro, listo para ser evaluado y exportado para inferencia en tiempo real.

In [10]:
print("\n" + "=" * 80)
print("FASE 4/4: FINE-TUNING FINAL - 1920px")
print("Objetivo: Estabilizar y pulir modelo para produccion (Augmentation Minimo)")
print("=" * 80)

fase3_best = f'{ruta_modelos}/fase3_alta_resolucion_1920px/weights/best.pt'
fase4_name = 'fase4_modelo_final_produccion'
fase4_done = f'{ruta_modelos}/{fase4_name}/results.png'

if not os.path.exists(fase3_best):
    print("[ERROR] No se encontro el modelo 'best.pt' de Fase 3. Ejecute primero la Fase 3.")

# Checar si ya termino al 100%
elif os.path.exists(fase4_done):
    print("[✓] FASE 4 YA COMPLETADA AL 100%. Modelo listo para produccion.")

# Empezar Fase 4 limpia y desde cero
else:
    print(f"Iniciando fine-tuning Fase 4 desde cero (Usando pesos Fase 3)...")
    inicio_fase4 = time.time()
    
    # Cargamos el modelo exactamente donde se quedó la Fase 3
    model_final = YOLO(fase3_best)
    
    # Arrancamos el entrenamiento
    results_final = model_final.train(
        data=yaml_path,
        epochs=30,
        imgsz=1920,
        device=device,
        project=ruta_modelos,
        name=fase4_name,
        batch=6,
        # Augmentation Minimo (Estabilizacion)
        degrees=5.0, translate=0.1, scale=0.9, perspective=0.0001, fliplr=0.5, flipud=0.0,
        hsv_h=0.005, hsv_s=0.3, hsv_v=0.2,
        mosaic=0.3, mixup=0.0, copy_paste=0.0, erasing=0.1, close_mosaic=5,
        optimizer='AdamW', lr0=0.0001, lrf=0.00001, momentum=0.95, weight_decay=0.0001,
        warmup_epochs=2.0, cos_lr=True,
        workers=8, amp=True, cache=False, # <--- 8 WORKERS ESTABLES
        val=True, plots=True, save=True, save_period=5, patience=50, verbose=True
    )
    
    tiempo_fase4 = (time.time() - inicio_fase4) / 60
    print(f"\n[✓] FASE 4 COMPLETADA en {tiempo_fase4:.2f} minutos")
    print(f"\n[✓] EL MODELO PARA PRODUCCION ESTA LISTO EN: {ruta_modelos}/{fase4_name}/weights/best.pt")


FASE 4/4: FINE-TUNING FINAL - 1920px
Objetivo: Estabilizar y pulir modelo para produccion (Augmentation Minimo)
Iniciando fine-tuning Fase 4 desde cero (Usando pesos Fase 3)...
Ultralytics 8.4.33 🚀 Python-3.11.15 torch-2.11.0+rocm7.2 CUDA:0 (AMD Instinct MI210, 65520MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/lustre/cursos/curso04/estudiante_64/epp-detection/data/raw/kaggle/sh17_dataset/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.1, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.005, hsv_s=0.3, hsv_v=0.2, imgsz=1920, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0

/lustre/cursos/curso04/estudiante_64/epp-detection/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/lustre/cursos/curso04/estudiante_64/epp-detection/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if 

optimizer: AdamW(lr=0.0001, momentum=0.95) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.00010312500000000001), 63 bias(decay=0.0)
Plotting labels to /lustre/cursos/curso04/estudiante_64/epp-detection/models/fase4_modelo_final_produccion/labels.jpg... 
Image sizes 1920 train, 1920 val
Using 8 dataloader workers
Logging results to /lustre/cursos/curso04/estudiante_64/epp-detection/models/fase4_modelo_final_produccion
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/30      1.07G      1.338      1.078       1.47         34       1920: 100% ━━━━━━━━━━━━ 1350/1350 3.4it/s 6:37<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 675/675 2.1it/s 5:17<0.1ss
                   all       8099      75994      0.623       0.53      0.559      0.322

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      11.5G

## 8. Fase 5: Evaluación Definitiva y Exportación a ONNX

Con el modelo estabilizado tras la fase de *Cooldown*, el paso final de nuestro pipeline es obtener las métricas de rendimiento definitivas y preparar el modelo para su despliegue en producción (Dashboard Operativo en Streamlit).

**Decisiones Técnicas para el Despliegue:**
1. **Validación en Resolución Nativa:** Ejecutamos la evaluación final a 1920px para obtener un reporte exacto de cómo se comportará el modelo en el entorno real, midiendo su Precisión, Recall y el mAP (Mean Average Precision).
2. **Conversión a ONNX (Open Neural Network Exchange):** Los archivos nativos de PyTorch (`.pt`) son excelentes para entrenar, pero pesados para la inferencia en tiempo real. Exportar a formato ONNX crea un grafo computacional agnóstico e hiperoptimizado que procesa los cuadros de video a mucha mayor velocidad.
3. **Cuantización a FP16 (`half=True`):** Aplicamos una técnica de reducción de precisión de punto flotante (de 32-bit a 16-bit). Esto reduce el tamaño del modelo a la mitad y acelera dramáticamente la inferencia (FPS) con una pérdida de precisión analítica prácticamente nula, lo cual es vital para el procesamiento de video en tiempo real.

In [12]:
print("\n" + "=" * 80)
print("FASE 5: EVALUACION FINAL Y EXPORTACION PARA STREAMLIT")
print("Objetivo: Generar reporte final y convertir a ONNX para maxima velocidad")
print("=" * 80)

# Buscamos el mejor modelo de la Fase 4
fase4_best = f'{ruta_modelos}/fase4_modelo_final_produccion/weights/best.pt'
export_path = f'{ruta_modelos}/fase4_modelo_final_produccion/weights/best.onnx'

if not os.path.exists(fase4_best):
    print("Esperando a que el modelo de la Fase 4 termine de entrenar...")
elif os.path.exists(export_path):
    print("EL MODELO YA FUE EVALUADO Y EXPORTADO A ONNX PREVIAMENTE.")
    print(f"Ruta lista para tu Dashboard: {export_path}")
else:
    print("Cargando el super modelo final para su ultima prueba...")
    model_produccion = YOLO(fase4_best)
    
    print("\n1. Ejecutando Validacion Final...")
    # Lo validamos con la resolucion nativa con la que entreno al final
    metricas = model_produccion.val(data=yaml_path, imgsz=1920, batch=4, device=device)
    
    print("\n" + "*" * 40)
    print("METRICAS FINALES DE PRODUCCION ")
    print("*" * 40)
    print(f"mAP50:     {metricas.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95:  {metricas.results_dict['metrics/mAP50-95(B)']:.4f}")
    print(f"Precision: {metricas.results_dict['metrics/precision(B)']:.4f}")
    print(f"Recall:    {metricas.results_dict['metrics/recall(B)']:.4f}")
    
    print("\n2. Exportando modelo a formato ONNX para Streamlit...")
    # Exportar a ONNX (half=True hace que el modelo pese la mitad sin perder precision)
    model_produccion.export(format='onnx', imgsz=1920, half=True, simplify=True)
    print(f"Tu modelo optimizado esta en: {export_path}")


FASE 5: EVALUACION FINAL Y EXPORTACION PARA STREAMLIT
Objetivo: Generar reporte final y convertir a ONNX para maxima velocidad
Cargando el super modelo final para su ultima prueba...

1. Ejecutando Validacion Final...
Ultralytics 8.4.33 🚀 Python-3.11.15 torch-2.11.0+rocm7.2 CUDA:0 (AMD Instinct MI210, 65520MiB)
Model summary (fused): 73 layers, 3,008,963 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2432.7±1616.6 MB/s, size: 1906.5 KB)
val: Scanning /lustre/cursos/curso04/estudiante_64/epp-detection/data/raw/kaggle/sh17_dataset/labels.cache... 8099 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8099/8099 2.0Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2025/2025 13.2it/s 2:34<0.1s
                   all       8099      75994      0.683      0.585      0.629      0.377
                Person       7617      13802      0.668      0.802      0.798      0.434
             